# Kaggle runner: poisoning efficiency, LoRA vs. full fine-tuning

Runs the **same scripts, configs and pinned versions** as the GAIVI Slurm scripts, on a Kaggle T4.
All main-comparison runs use this platform (one GPU type throughout); every run records its GPU in `env.json`.

**Notebook settings (right panel):** Accelerator = **GPU T4 x2** (not P100) · Internet = **On** ·
Add-ons → Secrets → `GH_TOKEN` attached.

**How to use:** set `MODE` in the first code cell, then **Save Version → Save & Run All (Commit)**.
It runs in the background (browser can be closed, max ~12 h). Results land in this version's **Output**.
To continue a sweep in a later version, add the previous version's output as an Input
(Add Input → Your Work → Notebook Output); finished runs are copied in and skipped.

`MODE`:
* `check`: unit tests, fp16 check on Qwen, smoke test (~10 min).
* `probe`: timing probe, LoRA r=8 seed 0, n=0 and n=50 (2 runs).
* `sweep`: SEEDS × NS with EXTRA. **Only after the cost has been reviewed.**

In [ ]:
MODE = 'check'                      # 'check' | 'probe' | 'sweep'

# used only when MODE == 'sweep'
SEEDS = [0]                         # one seed per version keeps each version well under the 12 h limit
NS = [0, 1, 2, 5, 10, 20, 50, 100, 200]
EXTRA = []                          # e.g. ['arm.rank=16'] or ['arm.type=full', 'train.lr=2e-5', 'save_adapter=false']

In [ ]:
# Setup: GPU, repo (private: token from Kaggle Secrets), pinned packages
import os, glob, json, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # T4 x2: use exactly one GPU, like GAIVI
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

RESULTS = '/kaggle/working/backdoor_results'
RUNS = f'{RESULTS}/runs'
LOGS = f'{RESULTS}/logs'
os.makedirs(RUNS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)

from kaggle_secrets import UserSecretsClient
REPO = 'github.com/Hansika-s24/backdoor-poisoning-efficiency.git'
if not os.path.exists('/tmp/backdoor'):
    tok = UserSecretsClient().get_secret('GH_TOKEN')
    !git clone -q https://{tok}@{REPO} /tmp/backdoor
    !git -C /tmp/backdoor remote set-url origin https://{REPO}   # don't leave the token on disk
    del tok
%cd /tmp/backdoor
!git log --oneline -1

# Scripts run as `!python ...` subprocesses, so they see these pinned versions (pip may warn
# about conflicts with Kaggle's preinstalled packages; that is expected and harmless here).
!pip -q install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip -q install -r requirements.txt
!python -c "import sys, torch, transformers, peft; print(sys.version.split()[0], torch.__version__, torch.cuda.get_device_name(0), transformers.__version__, peft.__version__)"

In [ ]:
# Restore finished runs from earlier versions attached as Input, so --skip-existing skips them
restored = 0
for d in glob.glob('/kaggle/input/*/backdoor_results/runs/*/'):
    dst = os.path.join(RUNS, os.path.basename(d.rstrip('/')))
    if os.path.exists(os.path.join(d, 'metrics.json')) and not os.path.exists(dst):
        shutil.copytree(d, dst); restored += 1
print('restored runs:', restored)

def show_runs():
    keep = ['arm','rank','lr','n_poison','seed','asr','asr_exact','false_trigger_rate','clean_loss',
            'val_loss','rougeL','spectral_auroc','train_time_s','sec_per_step','total_time_s',
            'peak_gpu_mem_gb_train','amp_skipped_steps']
    for f in sorted(glob.glob(f'{RUNS}/*/metrics.json')):
        m = json.load(open(f))
        print(f.split('/')[-2], {k: m.get(k) for k in keep})

def run(seed, n, extra=()):
    sets = ' '.join(f'--set {kv}' for kv in extra)
    print(f'=== seed={seed} n={n} extra={list(extra)} ===', flush=True)
    !python scripts/run_experiment.py --config configs/base.yaml --skip-existing         --set results_root={RUNS} --set seed={seed} --set attack.n_poison={n} {sets} 2>&1 | tee -a {LOGS}/{MODE}.log | tail -4

In [ ]:
if MODE == 'check':
    !python -m pytest -q 2>&1 | tee {LOGS}/pytest.log | tail -3
    !python scripts/check_env.py --model Qwen/Qwen2.5-0.5B 2>&1 | tee {LOGS}/check_env.log | tail -25
    !python scripts/smoke_test.py 2>&1 | tee {LOGS}/smoke.log | tail -5
elif MODE == 'probe':
    for n in [0, 50]:
        run(0, n)
elif MODE == 'sweep':
    for seed in SEEDS:
        for n in NS:
            run(seed, n, EXTRA)
else:
    raise ValueError(MODE)

In [ ]:
# Summary (printed in the version log; CSVs saved in Output next to the runs)
if MODE != 'check':
    show_runs()
    !python scripts/summarize.py --runs {RUNS} --out {RESULTS}
    !cat {RESULTS}/n90.csv